# 21｜不用预制 LSTM/CRF：手写 BiLSTM-CRF 序列标注

本笔记在单文件内重新实现 LSTM cell、双向变长扫描器、线性链 CRF 的 gold score、log-partition、NLL 与 Viterbi，再组合成 `BiLSTMCRF.forward`。我们用穷举验证动态规划，而不是仅凭训练 loss 猜实现正确。

## 1. 数据与状态合同

- token/tag/mask 形状都是 `[B,T]`，mask 必须是左对齐连续前缀且每条至少一个 token。
- emission 为 `[B,T,K]`；`transition[next_tag, previous_tag]`，不能混淆方向。
- START/END 不作为普通标签加入输出空间，而由独立参数表示。
- 约束采用布尔 `allowed_*`；非法 gold 路径立即拒绝。
- 标签采用 BIO：`0=O, 1=B-X, 2=I-X`。

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)

import hashlib
import io
import itertools
import math
import random

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20260730
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")
O, B_X, I_X = 0, 1, 2
TAG_NAMES = ["O", "B-X", "I-X"]

assert DEVICE.type == "cpu"
assert len(TAG_NAMES) == 3
assert torch.get_num_threads() == 1
print({"torch": torch.__version__, "tags": TAG_NAMES})

## 2. 单步 LSTM 与双向扫描

LSTM 使用 `i,f,g,o` 四门：$c_t=f_t\odot c_{t-1}+i_t\odot g_t$，$h_t=o_t\odot\tanh(c_t)$。双向网络并不是调用 `bidirectional=True`：前向按 $0\to T-1$ 扫描，反向按 $T-1\to0$ 扫描，两边在无效位置均冻结状态并把输出清零，最后拼成 `[B,T,2H]`。

In [ ]:
class ScratchLSTMCell(nn.Module):
    gate_order = "ifgo"
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size, self.hidden_size = input_size, hidden_size
        self.gates = nn.Linear(input_size + hidden_size, 4 * hidden_size)
    def forward(self, x_t, state):
        h, c = state
        i, f, g, o = self.gates(torch.cat([x_t, h], -1)).chunk(4, -1)
        i, f, o, g = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o), torch.tanh(g)
        c_new = f * c + i * g
        h_new = o * torch.tanh(c_new)
        return h_new, c_new

class ScratchBiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.forward_cell = ScratchLSTMCell(input_size, hidden_size)
        self.backward_cell = ScratchLSTMCell(input_size, hidden_size)

    def _scan(self, x, mask, cell, reverse=False):
        B, T, _ = x.shape
        h = x.new_zeros(B, self.hidden_size)
        c = x.new_zeros(B, self.hidden_size)
        outputs = [None] * T
        time_ids = range(T - 1, -1, -1) if reverse else range(T)
        for t in time_ids:
            active = mask[:, t].unsqueeze(-1)
            h_new, c_new = cell(x[:, t], (h, c))
            h, c = torch.where(active, h_new, h), torch.where(active, c_new, c)
            outputs[t] = torch.where(active, h, torch.zeros_like(h))
        return torch.stack(outputs, 1)

    def forward(self, x, mask):
        if x.ndim != 3 or mask.shape != x.shape[:2] or mask.dtype != torch.bool:
            raise ValueError("期望 x=[B,T,D] 与 bool mask=[B,T]")
        fw = self._scan(x, mask, self.forward_cell, reverse=False)
        bw = self._scan(x, mask, self.backward_cell, reverse=True)
        return torch.cat([fw, bw], -1)

bi0 = ScratchBiLSTM(5, 7)
mask0 = torch.tensor([[1, 1, 1, 0], [1, 1, 0, 0]], dtype=torch.bool)
bi_out0 = bi0(torch.randn(2, 4, 5), mask0)
assert bi_out0.shape == (2, 4, 14)
assert torch.count_nonzero(bi_out0[~mask0]) == 0
assert not any(isinstance(m, (nn.LSTM, nn.GRU, nn.RNN)) for m in bi0.modules())

## 3. 线性链 CRF 得分

路径 $y_{1:L}$ 的得分：

$$s(x,y)=a_{y_1}+\sum_{t=1}^{L}e_{t,y_t}+\sum_{t=2}^{L}A_{y_t,y_{t-1}}+b_{y_L}.$$

$a,b$ 是 START/END 分数，$A[next,prev]$ 是转移分数。条件概率为 $p(y|x)=\exp s(x,y)/Z(x)$，负对数似然是 $\log Z-s(x,y)$。所有动态规划都只更新 mask 为真的时间步。

## 4. log-partition：log-sum-exp 动态规划

初始化 $\alpha_1(k)=a_k+e_{1,k}$；递推：

$$\alpha_t(k)=e_{t,k}+\log\sum_j\exp(\alpha_{t-1}(j)+A_{k,j}).$$

最后 $\log Z=\log\sum_k\exp(\alpha_L(k)+b_k)$。约束通过把非法项替换为一个足够小的有限数；这里保留有限值以避免混合精度中 `inf-inf`。

In [ ]:
class LinearChainCRF(nn.Module):
    NEG = -1e4
    def __init__(self, num_tags, allowed_transitions=None, allowed_start=None, allowed_end=None):
        super().__init__()
        if num_tags <= 0:
            raise ValueError("num_tags 必须为正")
        self.num_tags = num_tags
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags))  # [next, prev]
        self.start = nn.Parameter(torch.empty(num_tags))
        self.end = nn.Parameter(torch.empty(num_tags))
        nn.init.uniform_(self.transitions, -0.1, 0.1)
        nn.init.uniform_(self.start, -0.1, 0.1)
        nn.init.uniform_(self.end, -0.1, 0.1)
        at = torch.ones(num_tags, num_tags, dtype=torch.bool) if allowed_transitions is None else allowed_transitions.bool()
        ast = torch.ones(num_tags, dtype=torch.bool) if allowed_start is None else allowed_start.bool()
        aend = torch.ones(num_tags, dtype=torch.bool) if allowed_end is None else allowed_end.bool()
        if at.shape != (num_tags, num_tags) or ast.shape != (num_tags,) or aend.shape != (num_tags,):
            raise ValueError("CRF 约束矩阵/向量 shape 与 num_tags 不一致")
        self.register_buffer("allowed_transitions", at)
        self.register_buffer("allowed_start", ast)
        self.register_buffer("allowed_end", aend)

    def _validate(self, emissions, mask, tags=None):
        if emissions.ndim != 3 or emissions.shape[-1] != self.num_tags:
            raise ValueError("emissions 必须为 [B,T,K]")
        if mask.shape != emissions.shape[:2] or mask.dtype != torch.bool:
            raise ValueError("mask 必须是 [B,T] bool")
        if bool((~mask[:, 0]).any()):
            raise ValueError("CRF 拒绝空序列")
        if bool(((~mask[:, :-1]) & mask[:, 1:]).any()):
            raise ValueError("mask 必须是连续左前缀")
        if tags is not None:
            if tags.shape != mask.shape:
                raise ValueError("tags shape 不匹配")
            if tags.dtype != torch.long:
                raise ValueError("tags 必须为 torch.long")
            if bool((((tags < 0) | (tags >= self.num_tags)) & mask).any()):
                raise ValueError("有效位置的 tag id 越界")

    def constrained(self):
        trans = self.transitions.masked_fill(~self.allowed_transitions, self.NEG)
        start = self.start.masked_fill(~self.allowed_start, self.NEG)
        end = self.end.masked_fill(~self.allowed_end, self.NEG)
        return trans, start, end

    def ensure_legal_path(self, mask):
        # 仅在布尔约束图上做可达性 DP，不让有限 NEG 伪装成一条低分合法路径。
        reachable = self.allowed_start.unsqueeze(0).expand(mask.shape[0], -1)
        for t in range(1, mask.shape[1]):
            next_reachable = (self.allowed_transitions.unsqueeze(0)
                              & reachable[:, None, :]).any(dim=-1)
            reachable = torch.where(mask[:, t, None], next_reachable, reachable)
        has_complete_path = (reachable & self.allowed_end.unsqueeze(0)).any(dim=-1)
        if bool((~has_complete_path).any()):
            bad_rows = (~has_complete_path).nonzero(as_tuple=False).flatten().tolist()
            raise ValueError(f"约束图中不存在完整合法路径，样本索引: {bad_rows}")
        return reachable

    def log_partition(self, emissions, mask):
        self._validate(emissions, mask)
        self.ensure_legal_path(mask)
        trans, start, end = self.constrained()
        alpha = start + emissions[:, 0]
        for t in range(1, emissions.shape[1]):
            scores = alpha[:, None, :] + trans[None, :, :] + emissions[:, t, :, None]
            next_alpha = torch.logsumexp(scores, dim=-1)
            alpha = torch.where(mask[:, t, None], next_alpha, alpha)
        return torch.logsumexp(alpha + end, dim=-1)

    def gold_score(self, emissions, tags, mask):
        self._validate(emissions, mask, tags)
        trans, start, end = self.constrained()
        first = tags[:, 0]
        if bool((~self.allowed_start[first]).any()):
            raise ValueError("gold 路径含非法 START 转移")
        score = start[first] + emissions[:, 0].gather(1, first[:, None]).squeeze(1)
        prev = first
        for t in range(1, emissions.shape[1]):
            cur, active = tags[:, t], mask[:, t]
            cur_safe = torch.where(active, cur, prev)  # padding tag 的具体填充值不应参与索引
            legal = self.allowed_transitions[cur_safe, prev]
            if bool((active & ~legal).any()):
                raise ValueError("gold 路径含非法标签转移")
            step = trans[cur_safe, prev] + emissions[:, t].gather(1, cur_safe[:, None]).squeeze(1)
            score = score + torch.where(active, step, torch.zeros_like(step))
            prev = cur_safe
        if bool((~self.allowed_end[prev]).any()):
            raise ValueError("gold 路径含非法 END 转移")
        return score + end[prev]

    def neg_log_likelihood(self, emissions, tags, mask):
        return (self.log_partition(emissions, mask) - self.gold_score(emissions, tags, mask)).mean()

assert LinearChainCRF(3).transitions.shape == (3, 3)
assert LinearChainCRF.NEG < -1000

## 5. Viterbi：把求和换成最大值

Viterbi 与 log-partition 共享状态图，但递推取 `max` 并保存每步 backpointer。变长 batch 在样本结束后冻结分数；回溯只使用各自有效长度。返回 Python 标签列表，便于后处理成实体 span。

In [ ]:
def crf_viterbi(self, emissions, mask):
    self._validate(emissions, mask)
    self.ensure_legal_path(mask)
    trans, start, end = self.constrained()
    score = start + emissions[:, 0]
    backpointers = []
    for t in range(1, emissions.shape[1]):
        candidates = score[:, None, :] + trans[None, :, :]
        best_score, best_prev = candidates.max(dim=-1)
        best_score = best_score + emissions[:, t]
        score = torch.where(mask[:, t, None], best_score, score)
        backpointers.append(best_prev)
    score = score + end
    best_last = score.argmax(-1)
    lengths = mask.sum(1).tolist()
    paths = []
    for b, length in enumerate(lengths):
        tag = int(best_last[b])
        path = [tag]
        for t in range(length - 1, 0, -1):
            tag = int(backpointers[t - 1][b, tag])
            path.append(tag)
        paths.append(list(reversed(path)))
    return paths, score.max(-1).values

LinearChainCRF.viterbi_decode = crf_viterbi
crf0 = LinearChainCRF(3)
em0 = torch.randn(2, 4, 3)
paths0, scores0 = crf0.viterbi_decode(em0, mask0)
assert list(map(len, paths0)) == [3, 2]
assert scores0.shape == (2,)
assert all(0 <= tag < 3 for path in paths0 for tag in path)

## 6. 穷举对照：给动态规划一个独立 oracle

长度 3、标签数 3 只有 $3^3=27$ 条路径，可以逐条算分。我们把穷举的 `logsumexp` 与最大路径分别对照 CRF 的 `log_partition` 和 Viterbi。这个测试能同时发现转移矩阵方向、START/END 和 emission 索引错误。

In [ ]:
def brute_force_scores(crf, emissions_1d):
    trans, start, end = crf.constrained()
    scored = []
    for path in itertools.product(range(crf.num_tags), repeat=emissions_1d.shape[0]):
        if not crf.allowed_start[path[0]] or not crf.allowed_end[path[-1]]:
            continue
        if any(not crf.allowed_transitions[path[t], path[t-1]] for t in range(1, len(path))):
            continue
        score = start[path[0]] + emissions_1d[0, path[0]]
        for t in range(1, len(path)):
            score = score + trans[path[t], path[t-1]] + emissions_1d[t, path[t]]
        score = score + end[path[-1]]
        scored.append((path, score))
    return scored

torch.manual_seed(SEED + 1)
tiny_crf = LinearChainCRF(3)
tiny_em = torch.randn(1, 3, 3)
tiny_mask = torch.ones(1, 3, dtype=torch.bool)
enumerated = brute_force_scores(tiny_crf, tiny_em[0])
brute_logz = torch.logsumexp(torch.stack([s for _, s in enumerated]), 0)
brute_path, brute_best = max(enumerated, key=lambda pair: float(pair[1]))
dp_logz = tiny_crf.log_partition(tiny_em, tiny_mask)[0]
dp_paths, dp_scores = tiny_crf.viterbi_decode(tiny_em, tiny_mask)

assert len(enumerated) == 27
assert torch.allclose(dp_logz, brute_logz, atol=1e-6)
assert tuple(dp_paths[0]) == brute_path
assert torch.allclose(dp_scores[0], brute_best, atol=1e-6)
assert dp_logz >= dp_scores[0]

## 7. BIO 合法转移

对单一实体类型，START 不能直接进入 `I-X`，`O -> I-X` 也非法；其他转移在这个简化协议中允许。复杂 NER 还要禁止 `B-PER -> I-ORG` 等跨类型连接。约束既用于分母也用于解码，并对 gold 路径做显式拒绝。

In [ ]:
allowed_trans = torch.ones(3, 3, dtype=torch.bool)
allowed_trans[I_X, O] = False       # transition[next=I, prev=O]
allowed_start = torch.tensor([True, True, False])
allowed_end = torch.tensor([True, True, True])
bio_crf = LinearChainCRF(3, allowed_trans, allowed_start, allowed_end)

illegal_tags = torch.tensor([[I_X, O]])
illegal_em = torch.randn(1, 2, 3)
try:
    bio_crf.gold_score(illegal_em, illegal_tags, torch.ones(1, 2, dtype=torch.bool))
    illegal_start_rejected = False
except ValueError:
    illegal_start_rejected = True

assert illegal_start_rejected
assert not bio_crf.allowed_start[I_X]
assert not bio_crf.allowed_transitions[I_X, O]
assert bio_crf.allowed_transitions[I_X, B_X]

## 8. BiLSTM-CRF 组合

Embedding 把 token 映射到 `[B,T,E]`，手写 BiLSTM 生成 `[B,T,2H]`，线性层产生 emission。训练调用 CRF NLL；推理调用 Viterbi。padding token 的 embedding 固定为零，但真正的边界仍由 mask 决定。

若词表 $V$、嵌入 $E$、单向隐藏维 $H$、标签数 $K$，参数量为：embedding $VE$；双向 LSTM $2	imes4H(E+H+1)$；emission $K(2H+1)$；CRF $K^2+2K$。

In [ ]:
class BiLSTMCRF(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_tags,
                 allowed_transitions, allowed_start, allowed_end):
        super().__init__()
        self.config = dict(vocab_size=vocab_size, embed_dim=embed_dim,
                           hidden_size=hidden_size, num_tags=num_tags)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.encoder = ScratchBiLSTM(embed_dim, hidden_size)
        self.emission = nn.Linear(2 * hidden_size, num_tags)
        self.crf = LinearChainCRF(num_tags, allowed_transitions, allowed_start, allowed_end)

    def emissions(self, tokens, mask):
        if tokens.shape != mask.shape:
            raise ValueError("tokens 与 mask shape 不一致")
        return self.emission(self.encoder(self.embedding(tokens), mask))

    def forward(self, tokens, mask, tags=None):
        emissions = self.emissions(tokens, mask)
        if tags is None:
            return self.crf.viterbi_decode(emissions, mask)[0]
        return self.crf.neg_log_likelihood(emissions, tags, mask)

model_probe = BiLSTMCRF(12, 8, 10, 3, allowed_trans, allowed_start, allowed_end)
probe_em = model_probe.emissions(torch.tensor([[2, 3, 0]]), torch.tensor([[1, 1, 0]], dtype=torch.bool))
parameter_breakdown21 = {
    "embedding": sum(p.numel() for p in model_probe.embedding.parameters()),
    "bilstm": sum(p.numel() for p in model_probe.encoder.parameters()),
    "emission": sum(p.numel() for p in model_probe.emission.parameters()),
    "crf": sum(p.numel() for p in model_probe.crf.parameters()),
}
assert probe_em.shape == (1, 3, 3)
assert not any(isinstance(m, (nn.LSTM, nn.GRU, nn.RNN)) for m in model_probe.modules())
assert parameter_breakdown21["crf"] == 3 * 3 + 2 * 3
assert sum(parameter_breakdown21.values()) == sum(p.numel() for p in model_probe.parameters())
print(parameter_breakdown21)

## 9. 小型 NER 受控过拟合集

token 语义：`1` 是 padding；实际 padding 仍使用 `0`。`2/3/4` 是普通词，`5/6` 是实体首词，`7/8` 是实体续词。每条序列左对齐，标签遵守 BIO。集合刻意覆盖单 token 实体、多 token 实体、句首/句中实体和不同长度。

这不是独立测试集。它仅用于证明 emission、CRF loss 与 Viterbi 能协同记住一个合法的小集合。

In [ ]:
token_rows = [
    [2, 5, 7, 3], [5, 7, 8, 4], [2, 3, 5], [6, 8, 4, 2, 3],
    [4, 2], [3, 6, 7], [5, 2, 3, 4], [2, 6, 8, 3, 4],
]
tag_rows = [
    [O, B_X, I_X, O], [B_X, I_X, I_X, O], [O, O, B_X], [B_X, I_X, O, O, O],
    [O, O], [O, B_X, I_X], [B_X, O, O, O], [O, B_X, I_X, O, O],
]
max_t = max(map(len, token_rows))
tokens21 = torch.zeros(len(token_rows), max_t, dtype=torch.long)
tags21 = torch.zeros_like(tokens21)
mask21 = torch.zeros_like(tokens21, dtype=torch.bool)
for i, (tokens, tags) in enumerate(zip(token_rows, tag_rows)):
    tokens21[i, :len(tokens)] = torch.tensor(tokens)
    tags21[i, :len(tags)] = torch.tensor(tags)
    mask21[i, :len(tokens)] = True

assert tokens21.shape == tags21.shape == mask21.shape == (8, 5)
assert mask21.sum(1).tolist() == list(map(len, token_rows))
assert not ((tags21 == I_X) & ~mask21).any()
assert all(tags[0] != I_X for tags in tag_rows)

## 10. NLL 优化、梯度与裁剪

Adam 最小化 batch 平均 CRF NLL。动态规划全程可微，梯度应流向 embedding、两个方向的 LSTM、emission 和转移参数。首步检查非零有限梯度，随后裁剪全局范数。

In [ ]:
torch.manual_seed(SEED + 2)
model21 = BiLSTMCRF(12, 12, 14, 3, allowed_trans, allowed_start, allowed_end)
opt21 = torch.optim.Adam(model21.parameters(), lr=0.025)
losses21 = []
model21.train()
for step in range(220):
    opt21.zero_grad(set_to_none=True)
    loss = model21(tokens21, mask21, tags21)
    loss.backward()
    if step == 0:
        named_grads = {n: p.grad for n, p in model21.named_parameters() if p.grad is not None}
        assert named_grads
        assert all(torch.isfinite(g).all() for g in named_grads.values())
        assert sum(float(g.abs().sum()) for g in named_grads.values()) > 0
        assert model21.crf.transitions.grad.abs().sum() > 0
    torch.nn.utils.clip_grad_norm_(model21.parameters(), 1.0)
    opt21.step()
    losses21.append(float(loss.detach()))

model21.eval()
with torch.no_grad():
    pred_paths21 = model21(tokens21, mask21)

gold_paths21 = [tags[:len(tokens)] for tags, tokens in zip(tag_rows, token_rows)]
token_correct = sum(p == g for pred, gold in zip(pred_paths21, gold_paths21) for p, g in zip(pred, gold))
token_total = sum(map(len, gold_paths21))
token_acc21 = token_correct / token_total

assert losses21[-1] < losses21[0] * 0.1
assert token_acc21 >= 0.97
assert list(map(len, pred_paths21)) == list(map(len, gold_paths21))
print({"loss_first": losses21[0], "loss_last": losses21[-1], "controlled_token_acc": token_acc21})

## 11. 从 BIO 标签恢复 span，并计算实体级 F1

token accuracy 会被大量 `O` 稀释，所以 NER 主要看实体 span 的精确匹配。简化规则：`B-X` 开启实体，连续 `I-X` 延长；孤立 `I-X` 视为非法并拒绝。计算 micro precision/recall/F1，而不是把每句 F1 简单平均。

In [ ]:
def bio_to_spans(tags):
    spans, start = [], None
    for i, tag in enumerate(list(tags) + [O]):
        if tag == B_X:
            if start is not None:
                spans.append((start, i, "X"))
            start = i
        elif tag == I_X:
            if start is None:
                raise ValueError("孤立 I-X 不是合法 BIO")
        else:
            if start is not None:
                spans.append((start, i, "X"))
                start = None
    return spans

gold_spans = {(i, *span) for i, tags in enumerate(gold_paths21) for span in bio_to_spans(tags)}
pred_spans = {(i, *span) for i, tags in enumerate(pred_paths21) for span in bio_to_spans(tags)}
tp = len(gold_spans & pred_spans)
precision21 = tp / len(pred_spans) if pred_spans else 0.0
recall21 = tp / len(gold_spans) if gold_spans else 0.0
span_f1_21 = 2 * precision21 * recall21 / (precision21 + recall21) if precision21 + recall21 else 0.0

assert gold_spans
assert 0.0 <= precision21 <= 1.0
assert 0.0 <= recall21 <= 1.0
assert span_f1_21 >= 0.95
assert bio_to_spans([O, B_X, I_X, O]) == [(1, 3, "X")]
print({"span_precision": precision21, "span_recall": recall21, "span_f1": span_f1_21})

## 12. 失败反例与数值边界

- 把转移写成 `[prev,next]` 却按 `[next,prev]` 索引：loss 仍可能下降，但 Viterbi 路径错误。
- 分母允许非法路径、分子拒绝非法路径：概率空间不一致。
- 约束图对某个序列长度根本没有 START→…→END 路径，却仍用有限大负数继续解码：会把非法低分路径伪装成结果；必须先做布尔可达性检查并 fail closed。
- padding 后又出现有效 token：冻结动态规划会静默丢失中间洞，因此必须拒绝非前缀 mask。
- 用逐 token softmax 替代 CRF：无法建模标签转移约束。
- 在 fp16 直接使用真正的负无穷并做不稳定运算：容易出现 NaN；生产实现应验证 dtype 策略。
- 仅看 token accuracy：全预测 O 也可能很高，必须报告 span F1。

In [ ]:
def rejected(fn):
    try:
        fn()
        return False
    except ValueError:
        return True

empty_mask = torch.tensor([[False, False]])
hole_mask = torch.tensor([[True, False, True]])
no_start_crf = LinearChainCRF(2, allowed_start=torch.zeros(2, dtype=torch.bool))
dead_end_crf = LinearChainCRF(
    2, allowed_transitions=torch.zeros(2, 2, dtype=torch.bool),
    allowed_start=torch.tensor([True, False]), allowed_end=torch.ones(2, dtype=torch.bool),
)
assert rejected(lambda: model21.crf.log_partition(torch.randn(1, 2, 3), empty_mask))
assert rejected(lambda: model21.crf.log_partition(torch.randn(1, 3, 3), hole_mask))
assert rejected(lambda: no_start_crf.log_partition(
    torch.randn(1, 1, 2), torch.ones(1, 1, dtype=torch.bool)))
assert rejected(lambda: no_start_crf.viterbi_decode(
    torch.randn(1, 1, 2), torch.ones(1, 1, dtype=torch.bool)))
assert torch.isfinite(dead_end_crf.log_partition(
    torch.randn(1, 1, 2), torch.ones(1, 1, dtype=torch.bool))).all()
assert rejected(lambda: dead_end_crf.log_partition(
    torch.randn(1, 2, 2), torch.ones(1, 2, dtype=torch.bool)))
assert rejected(lambda: dead_end_crf.viterbi_decode(
    torch.randn(1, 2, 2), torch.ones(1, 2, dtype=torch.bool)))
assert rejected(lambda: bio_to_spans([I_X, O]))
assert rejected(lambda: LinearChainCRF(3, torch.ones(2, 2, dtype=torch.bool)))
assert rejected(lambda: model21.crf.gold_score(torch.randn(1, 2, 3),
                                                torch.zeros(1, 2),
                                                torch.ones(1, 2, dtype=torch.bool)))
assert rejected(lambda: model21.emissions(torch.ones(1, 2, dtype=torch.long),
                                          torch.ones(1, 3, dtype=torch.bool)))
assert torch.isfinite(model21.crf.transitions).all()

## 13. 制品合同与指纹

除了网络维度，必须固化 tag-to-id、BIO 约束矩阵、转移方向、padding id、mask 语义和解码算法版本。任何一项变化都会让相同权重产生不同实体边界。下面在内存保存并重新加载，避免依赖外部文件。

In [ ]:
manifest21 = {
    "artifact": "bilstm_crf_from_scratch",
    "schema_version": 1,
    "config": model21.config,
    "tags": TAG_NAMES,
    "padding_id": 0,
    "lstm_gate_order": ScratchLSTMCell.gate_order,
    "transition_layout": "transition[next_tag, previous_tag]",
    "mask_contract": "non-empty contiguous left prefix",
    "torch_version": torch.__version__,
}
buf21 = io.BytesIO()
torch.save({"manifest": manifest21, "state_dict": model21.state_dict()}, buf21)
artifact21 = buf21.getvalue()
sha21 = hashlib.sha256(artifact21).hexdigest()
buf21.seek(0)
loaded21 = torch.load(buf21, map_location="cpu", weights_only=False)
clone21 = BiLSTMCRF(**loaded21["manifest"]["config"],
                    allowed_transitions=allowed_trans,
                    allowed_start=allowed_start,
                    allowed_end=allowed_end)
clone21.load_state_dict(loaded21["state_dict"])
clone21.eval()
with torch.no_grad():
    clone_paths21 = clone21(tokens21, mask21)

assert len(sha21) == 64
assert clone_paths21 == pred_paths21
assert loaded21["manifest"]["transition_layout"].startswith("transition[next_tag")
assert set(clone21.state_dict()) == set(model21.state_dict())
print({"sha256": sha21[:16] + "…", "bytes": len(artifact21)})

## 14. 生产替换、安全与观测

手写 Python 时间循环适合学习和小规模校验，不适合长序列吞吐。生产可换成官方融合 BiLSTM 与经过审计的 CRF，但必须保留穷举 oracle、变长 batch、非法路径和 span 指标回归。

服务入口限制最大序列长度、词表范围和 batch；不加载不可信 pickle；记录未知 token 率、长度/截断分布、非法 BIO 率、span 数量、各标签比例、NLL、梯度范数、延迟和制品哈希。涉及人名等敏感实体时，日志应去标识化并做租户隔离。

## 15. 原始论文与官方文档

- Hochreiter & Schmidhuber, *Long Short-Term Memory* (1997)：https://doi.org/10.1162/neco.1997.9.8.1735
- Lafferty et al., *Conditional Random Fields* (2001)：https://repository.upenn.edu/cis_papers/159/
- Huang et al., *Bidirectional LSTM-CRF Models for Sequence Tagging* (2015)：https://arxiv.org/abs/1508.01991
- PyTorch `nn.Module`：https://pytorch.org/docs/stable/generated/torch.nn.Module.html
- PyTorch `logsumexp`：https://pytorch.org/docs/stable/generated/torch.logsumexp.html

模型思路与概率公式来自上述论文；代码和测试为本笔记重新实现。

In [ ]:
# 最终合同检查：分数关系、参数梯度、约束与指纹。
with torch.no_grad():
    final_em21 = model21.emissions(tokens21, mask21)
    final_logz21 = model21.crf.log_partition(final_em21, mask21)
    final_gold21 = model21.crf.gold_score(final_em21, tags21, mask21)
    padded_tags21 = tags21.clone()
    padded_tags21[~mask21] = -100
    padded_gold21 = model21.crf.gold_score(final_em21, padded_tags21, mask21)

assert final_logz21.shape == final_gold21.shape == (8,)
assert torch.all(final_logz21 + 1e-5 >= final_gold21)
assert torch.allclose(final_gold21, padded_gold21)
assert losses21[-1] >= -1e-5
assert token_total == int(mask21.sum())
assert manifest21["tags"] == TAG_NAMES
assert manifest21["lstm_gate_order"] == "ifgo"
assert sha21 == hashlib.sha256(artifact21).hexdigest()
assert all(torch.isfinite(p).all() for p in model21.parameters())
assert pred_paths21 == clone_paths21
print("Notebook 21：全部合同测试通过。")